# TP4 — ACP, k-means, classification hiérarchique
**CNFSDP — Introduction au Machine Learning** · L3 S5 · 2026-2027 · Rois Céti DIMBAMBA · chapitre 4 · 45 min en salle

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rdimbamba/ml-cnfsdp/blob/main/notebooks/TP4.ipynb)

Partie A : ACP à la main puis indice de richesse. Partie B : k-means et choix du nombre de groupes. Partie C : CAH et forêt d'isolement.
Les ménages sont **fictifs** : ils sont simulés au format des biens relevés par les enquêtes DHS, ce qui permet de connaître la « vraie » richesse et de juger l'indice. Aucun téléchargement n'est nécessaire.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
VERT, OR, GRIS = "#126B2A", "#B8860B", "#8A8A8A"   # couleurs du cours


Fonction de simulation des ménages (à exécuter sans la modifier) :

In [ ]:
def simuler_menages(n=2000, graine=0):
    """Ménages fictifs au format des biens DHS : 1 = le ménage possède le bien."""
    rng = np.random.default_rng(graine)
    urbain = rng.random(n) < 0.4
    richesse = rng.standard_normal(n) + 1.2 * urbain            # richesse latente (inobservée)
    sig = lambda t: 1 / (1 + np.exp(-t))
    biens = {  # (constante, effet richesse, effet urbain)
        "electricite": (-0.5, 2.0, 1.5), "eau_courante": (-1.5, 1.8, 1.5), "toilettes": (-1.0, 1.6, 0.8),
        "sol_fini": (-0.3, 1.8, 0.8), "radio": (0.3, 0.8, 0.0), "television": (-1.5, 2.2, 0.8),
        "refrigerateur": (-2.8, 2.2, 0.5), "telephone": (1.0, 1.5, 0.5), "moto": (-1.5, 0.8, -0.5),
        "voiture": (-4.5, 2.0, 0.3), "terre_agricole": (0.8, -0.3, -2.5), "betail": (0.0, -0.2, -2.0)}
    df = pd.DataFrame({b: (rng.random(n) < sig(a + r * richesse + u * urbain)).astype(int)
                       for b, (a, r, u) in biens.items()})
    return df, richesse, urbain

biens, richesse, urbain = simuler_menages()
biens.head()

## Partie A — Analyse en composantes principales
### Question 1 — ACP à la main
Quatre ménages, scores centrés de logement et d'équipement : $(-2,-1)$, $(-1,-2)$, $(1,2)$, $(2,1)$. Calculer $S=\frac14X^\top X$, ses valeurs et vecteurs propres (`np.linalg.eigh`), puis tracer $u^\top Su$ pour $u=(\cos\theta,\sin\theta)$, $\theta\in[0,180°]$. Retrouver le maximum $4{,}5$ en $45°$ et la part d'inertie de $90\,\%$.

In [ ]:
X4 = np.array([[-2, -1], [-1, -2], [1, 2], [2, 1]], float)
# À compléter : S, np.linalg.eigh(S), tracé de u'Su selon l'angle, maximum et part d'inertie


### Question 2 — L'indice de richesse
Faire une ACP normée des 12 biens (`StandardScaler` puis `PCA`). Afficher l'éboulis des valeurs propres et les corrélations de chaque bien avec les deux premières composantes ($\sqrt{\lambda_k}\,u_{jk}$). Construire l'indice de richesse (composante 1, orientée pour croître avec l'électricité), ses quintiles, et le taux d'équipement en réfrigérateur par quintile. Comparer l'indice à la richesse cachée `richesse`.

In [ ]:
# À compléter : Z standardisé, PCA().fit(Z), éboulis, corrélations biens-axes,
# indice = composante 1 orientée, quintiles (pd.qcut), réfrigérateur par quintile, corrélation avec richesse


**À commenter :** combien d'axes garder (règle $\lambda>1$) ? Que représente l'axe 1 ? Quels biens tirent l'axe 2 ?

## Partie B — k-means
### Question 3 — L'algorithme de Lloyd en NumPy
Programmer k-means ($k=2$) pour les revenus $1, 2, 4, 6, 9, 12$ : affectation au centre le plus proche, puis centres = moyennes, jusqu'à stabilité, en affichant $W$ à chaque itération. Partir des centres $1$ et $2$, puis $3$ et $11$. Quelle partition a la plus petite inertie ?

In [ ]:
x = np.array([1, 2, 4, 6, 9, 12.])

def lloyd(x, centres):
    # À compléter : boucle affectation / mise à jour, affichage de W, arrêt quand les centres ne bougent plus
    pass

lloyd(x, [1, 2]); print()
lloyd(x, [3, 11])


### Question 4 — Combien de groupes de ménages ?
Sur les deux premières composantes, calculer l'inertie intra $W(k)$ et la silhouette moyenne de `KMeans(n_init=10)` pour $k=2,\dots,8$ ; tracer les deux courbes. Retenir $k=3$, puis décrire chaque groupe par son effectif et ses taux d'équipement (électricité, voiture, bétail).

In [ ]:
# À compléter : C = deux premières composantes, boucle sur k (inertia_, silhouette_score), deux tracés,
# puis KMeans à 3 groupes et profil des groupes (groupby sur km.labels_)


## Partie C — Classification hiérarchique et forêt d'isolement
### Question 5 — CAH de Ward sur les six revenus
Avec `linkage(x.reshape(-1, 1), method="ward")`, tracer le dendrogramme. scipy affiche une hauteur $h=\sqrt{2\Delta}$ : retrouver les hausses d'inertie $\Delta=h^2/2$ du cours ($0{,}5$ ; $2$ ; $4{,}5$ ; $12{,}25$ ; $70{,}08$). Pour la partition en 2 groupes, vérifier $T=W+B$.

In [ ]:
L = linkage(x.reshape(-1, 1), method="ward")
# À compléter : dendrogramme, hausses d'inertie L[:, 2]**2 / 2, puis T, W et B pour {1,2,4,6} et {9,12}


### Question 6 — Les ménages atypiques
Ajuster `IsolationForest(contamination=0.01)` sur les biens standardisés. Combien de ménages sont signalés ? Comparer leurs taux d'équipement à ceux de l'ensemble des ménages : qu'ont-ils d'inhabituel ?

In [ ]:
# À compléter : IsolationForest(contamination=0.01, random_state=0).fit(Z), ménages prédits -1, comparaison des moyennes


**Pour aller plus loin :** refaire l'ACP et une CAH sur des indicateurs des pays africains (Banque mondiale, World Development Indicators, https://data.worldbank.org), par exemple PIB par habitant, espérance de vie, scolarisation et accès à l'électricité.